# 台灣國道連假塞車程度與旅行時間區間預測｜最終分析（Google Colab 可重現）

本 Notebook 是 **Step 2 建模＋敘事圖表** 的單一入口。從 `data/processed/` 的 15 分鐘走廊資料出發，
搭配天氣、日曆、Google Trends 等外部變數，訓練模型 A／B／C 並以 **2025 年連假** 作時序測試。

## 資料來源與設計

| 項目 | 內容 |
| --- | --- |
| 車流主檔 | `data/processed/corridor_15m.parquet`（TDCS M05A eTag，5→15 分鐘，里程加權車速與旅行時間） |
| 外部變數 | `data/external/` 的氣象（Open-Meteo ERA5）、人事行政總處日曆、Google Trends（真實抓取，失敗即標缺席） |
| 時間切分 | 訓練 2023–2024、測試 2025；**同一年份內不隨機打散**，全部按時序向前預測 |
| 防洩漏 | 特徵僅用「可用資訊時點」：事件倒數、前序時段、前期天氣；2025 標籤不進任何特徵／調參；消融比較用同一測試集 |
| 標籤 | 四等級壅塞：0 順暢（≥80 km/h）、1 車多（60–80）、2 壅塞（40–60）、3 嚴重壅塞（<40） |
| 模型 A | 基準：日曆＋時段＋走廊 |
| 模型 B | A ＋ 天氣／車流前兆特徵 |
| 模型 C | B ＋ 黃金三角：5C 長連假提早北返、3A 天氣剪刀差、5A 請假攻略槓桿；另輸出 P50／P90 |

## Outputs

`outputs/phase2/`：`ablation_metrics.csv`、`ablation_report.md`、`predictions_2025.csv`、
`feature_importance.csv`、`confusion_matrix_model_C.csv`、`summary.json`。

`outputs/figures/`：`fig1_holiday_overview.png`、`fig2_ablation_comparison.png`、
`fig3_confusion_matrix_modelC.png`、`fig4_travel_time_quantile.png`、`fig5_feature_importance.png`（全部 dpi=300）。

> **可重現性**：所有數字皆由上述真實資料計算；缺檔時程式會印出明確訊息，不會補假資料。請從上到下 Run All。

In [ ]:
# ── 第 1 格（第一個程式格）：中文字型 Noto Sans TC ──────────────────────────
# Colab 預設沒有中文字型；下載 Static TC subset 並用 fontManager.addfont() 註冊。
# 若無網路或本機已有微軟正黑體，則try/except 降級，不中斷流程。
import os
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
from matplotlib import font_manager

FONT_URLS = {
    "Regular": "https://raw.githubusercontent.com/notofonts/noto-cjk/main/Sans/SubsetOTF/TC/NotoSansTC-Regular.otf",
    "Bold": "https://raw.githubusercontent.com/notofonts/noto-cjk/main/Sans/SubsetOTF/TC/NotoSansTC-Bold.otf",
}
CANDIDATES = ("Noto Sans TC", "Microsoft JhengHei", "Noto Sans CJK TC", "PingFang TC")


def installed_fonts():
    out = {}
    try:
        for f in font_manager.fontManager.ttflist:
            out.setdefault(f.name, set()).add(str(f.weight))
    except Exception:
        pass
    return out


def set_chinese_font():
    fonts = installed_fonts()
    chosen = next((n for n in CANDIDATES if n in fonts), None)
    need_download = chosen is None or (
        chosen == "Noto Sans TC" and not {"400", "700"}.issubset(fonts.get(chosen, set()))
    )
    if need_download and os.environ.get("FINAL_FIGURES_NO_FONT_DOWNLOAD") != "1":
        system_dir = Path("/usr/share/fonts/truetype/noto")
        target_dir = (system_dir if system_dir.parent.is_dir()
                      and os.access(str(system_dir.parent), os.W_OK)
                      else Path.home() / ".fonts")
        target_dir.mkdir(parents=True, exist_ok=True)
        for weight, url in FONT_URLS.items():
            target = target_dir / f"NotoSansTC-{weight}.otf"
            try:
                if not target.exists():
                    with urlopen(url, timeout=60) as response:
                        payload = response.read()
                    if payload[:4] not in (b"OTTO", b"\x00\x01\x00\x00"):
                        raise ValueError("不是有效的字型檔")
                    target.write_bytes(payload)
                font_manager.fontManager.addfont(str(target))
                print(f"[字型] 已註冊 {target}")
            except Exception as exc:  # 離線／權限不足時靜默降級
                print(f"[字型] 下載 {weight} 失敗（{exc}）")
        fonts = installed_fonts()
        chosen = next((n for n in CANDIDATES if n in fonts), None)
    if chosen is None:
        print("[字型] 警告：找不到中文字型，中文可能顯示為方框")
    plt.rcParams["font.family"] = chosen or "DejaVu Sans"
    plt.rcParams["font.sans-serif"] = [n for n in CANDIDATES] + ["DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 300
    plt.rcParams["savefig.dpi"] = 300
    return chosen or "DejaVu Sans"


FONT = set_chinese_font()
print("使用中文字型：", FONT)

In [ ]:
# ── 取得 repo ───────────────────────────────────────────────────────────────
# 方式 A（預設）：把整個 repo 上傳到 Colab 工作目錄，或先 git clone 後直接 Run All。
# 方式 B（Google Drive）：取消下面三行註解，指向你自己的資料夾。
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = Path('/content/drive/MyDrive/holiday_traffic')
# 方式 C（git clone）：取消註解並替換 repo URL。
# !git clone https://github.com/<USER>/<REPO>.git /content/holiday_traffic
# BASE_DIR = Path('/content/holiday_traffic')

import os
from pathlib import Path

BASE_DIR = Path.cwd()
for candidate in (Path.cwd(), Path('/content'), Path('/content/holiday_traffic'),
                  Path('/content/drive/MyDrive/holiday_traffic')):
    if (candidate / 'phase2_model.py').exists():
        BASE_DIR = candidate
        break
os.chdir(BASE_DIR)
print('工作目錄：', Path.cwd())
print('找到 phase2_model.py：', Path('phase2_model.py').exists())

In [ ]:
# ── 安裝相依套件 ────────────────────────────────────────────────────────────
%pip install -q lightgbm shap scikit-learn matplotlib pandas numpy pyarrow
print('套件安裝完成')

In [ ]:
# ── Step 2 建模 + 歷史時序驗證：產生 outputs/phase2/ 與 outputs/walk_forward/ ──
import subprocess
import sys
from pathlib import Path

def run_script(name: str) -> bool:
    script = Path(name)
    if not script.exists():
        print(f'找不到 {name}：請先上傳該檔，或取消上一格的方式 B/C 註解。')
        return False
    proc = subprocess.run([sys.executable, str(script)], text=True,
                          capture_output=True, cwd=str(Path.cwd()),
                          encoding='utf-8', errors='replace')
    print((proc.stdout or '')[-6000:])
    if (proc.stderr or '').strip():
        print((proc.stderr or '')[-3000:], file=sys.stderr)
    if proc.returncode != 0:
        print(f'警告：{name} 結束碼 {proc.returncode}；後續將依實際存在的檔案產生或缺檔提示。')
        return False
    print(f'{name} 執行完成。')
    return True

run_script('phase2_model.py')
run_script('walk_forward_validation.py')  # 2023→2024、2023–2024→2025 兩組時序驗證
run_script('confidence_intervals.py')     # 測試日區塊 bootstrap 95% CI


In [ ]:
# ── 產生 5 張中文敘事圖 ─────────────────────────────────────────────────────
import subprocess
import sys
from pathlib import Path

script = Path('src/final_figures.py')
if not script.exists():
    print('找不到 src/final_figures.py，請確認 repo 上傳完整。')
else:
    proc = subprocess.run([sys.executable, str(script)], text=True,
                          capture_output=True, cwd=str(Path.cwd()),
                          encoding='utf-8', errors='replace')
    print(proc.stdout or '')
    if (proc.stderr or '').strip():
        print((proc.stderr or '')[-3000:], file=sys.stderr)
    print(f'圖表產生器結束碼：{proc.returncode}')


In [ ]:
# ── 顯示五張圖、消融報告與 walk-forward 對照 ────────────────────────────────
from IPython.display import Image, Markdown, display
from pathlib import Path

figures = [
    'fig1_holiday_overview.png',
    'fig2_ablation_comparison.png',
    'fig3_confusion_matrix_modelC.png',
    'fig4_travel_time_quantile.png',
    'fig5_feature_importance.png',
]
for name in figures:
    path = Path('outputs/figures') / name
    if path.exists():
        print(f'■ {name}（dpi=300）')
        display(Image(filename=str(path), width=1200))
    else:
        print(f'□ 尚未產生：{path}（詳見上一格訊息）')

for report_name in ('outputs/phase2/ablation_report.md',
                    'outputs/phase2/confidence_intervals.md',
                    'outputs/walk_forward/walk_forward_report.md'):
    report = Path(report_name)
    if report.exists():
        display(Markdown(report.read_text(encoding='utf-8')))
    else:
        print(f'尚未產生 {report_name}；可執行 python phase2_model.py / python walk_forward_validation.py')


## 八、未來展望與研究限制

### 8.1 研究範圍：僅限國道

本專題**只處理國道**（TDCS M05A eTag 門架），不推論省道、市區道路或交流道層級。
模型覆蓋國5南向、國5北向、國1南向三條走廊；國道其餘方向可沿同一管線擴充，國2 因無 eTag 資料而例外。

### 8.2 未納入變數與推論限制

本版模型以「真實存在且可取得」的資料為前提。下列變數**未納入**，相關推論必須保守解讀：

| 變數 | 現況 | 推論限制 |
| --- | --- | --- |
| `airport`（桃園機場每日進出旅次） | 未接入 | 國1 桃園－新竹路段的機場旅次需求衝擊無法單獨歸因；尖峰可能被誤歸於連假效應。 |
| `hsr`（高鐵連假增班／載客率） | 未接入 | 可能存在跨運具替代（國道轉高鐵），模型可能高估國1 平行走廊的需求，替代比率效果不可論述。 |
| `t9_hourly`（台9線每小時流量） | 非研究範圍 | 本專題僅做國道，**不需要**省道檔；台9線分流只能列為未來對照組，不影響國道結論。 |
| `mega_events`（大型活動人流） | 未接入 | 局部時段尖峰無法歸因於演唱會、展演或體育賽事，可能殘留在誤差項。 |
| `M06A`（eTag 個別車輛資料） | 未接入 | 目前僅用 M05A 區間平均車速與流量，無法描述旅行時間分布、車種異質性與個別車輛跟車行為；P90 僅為分位數模型輸出。 |
| 國2 VD（車輛偵測器） | 未接入 | `N2_WB`／`N2_EB` 走廊定義保留，但 TDCS M05A/M06A 不含國2 資料；**國2 走廊不作任何推論**，圖表若出現僅代表定義存在。 |

此外，四等級標籤為 15 分鐘區間平均車速的門檻化結果，會低估短暫停等；2025 測試僅涵蓋少數連假，
P90 覆蓋率與 MAE 外推到 2026 仍需重新校準。Google Trends 若抓取失敗即標為缺席，不以插值填補。

快速補丁的誠實揭露：1A 修復後為**反假設驗證結果**（ρ=0.348, p=0.157），新抓關鍵字中「國道5號」全為 0、
「雪隧」僅少數天非零，故搜尋焦慮指數主要由「高乘載／天氣」構成；2A 效果弱且不顯著，2B 僅邊緣顯著。
交管表為依規則重建之索引，非高公局官方逐日公告，2C/2D 仍維持資料不足。

### 8.3 歷史資料驗證（walk-forward，如實揭露）

另以兩組時序切分重跑相同管線：**2023→2024** 與 **2023–2024→2025**，完整對照見
`outputs/walk_forward/walk_forward_report.md`。兩 fold 的 Macro F1、Log Loss、Brier、
P50 MAE 一致呈現 **C > B > A**，支持跨域增強不是單一年份偶然；但**嚴重壅塞 Recall 不一致**：
Fold 2024 由 B 較高（0.5297 vs C 0.4770），Fold 2025 才由 C 領先（0.5429 vs 0.4544）。
此外 P90 覆蓋率兩 fold 為 0.850–0.894，未穩定達 0.90。故不宣稱模型 C 六項指標一致領先，
也不把歷史時序驗證誤稱作 2026 前瞻外部驗證。

可信度另以 **2000 次測試日區塊 bootstrap** 量化（見 `outputs/phase2/confidence_intervals.md`）：
2025 整體等級命中 79.3%（最大宗 baseline 66.3%），2024 為 73.7%（baseline 62.1%）；
C 優於 B 的勝率在 2025 八項指標中七項為 100%（P90 為 93.1%），但 2024 嚴重 Recall 僅 4.2%（B 較高）。
P90 CI 2025 為 [0.867, 0.913]、2024 為 [0.838, 0.885]，僅 2025 與 0.90 相容。

### 8.4 未來展望（國道範圍內）

1. 在同一管線擴充 N1_NB、N3_SB、N3_NB（M05A 已有資料），讓覆蓋從三條走廊擴大為國1/3/5 主要雙向。
2. 補齊國2 VD 與 M06A 後再跑消融，驗證黃金三角與交管邊界是否仍為主要貢獻。
3. 將 P50/P90 擴充到完整分位數集合（P10/P50/P90），並做可靠性圖與 Pinball Loss 評估。
4. 引進空間圖神經網絡處理走廊間的連鎖壅塞，並以 2026 連假做真正的前瞻外部驗證。
5. 將 15 分鐘解析度下沉到 5 分鐘，搭配 M06A 單車資料評估「臨界減慢前兆訊號」的預警提前量。
